# 12.2 同樣 1600 個數字，為什麼可能長短不同？


拿到 1600 個聲音樣本，還不能知道錄了多久。若每秒取 16000 點，它長 0.1 秒；每秒取 8000 點，則長 0.2 秒。這個「每秒多少點」叫取樣率，是讀懂時間軸的必要資料。

取樣率也限制能分清多快的起伏：它的一半稱**奈奎斯特界線**。16000 Hz 的界線是8000 Hz，8000 Hz 的界線是4000 Hz。理想條件下，高於界線的變化可能混成較低頻率，不能由這串樣本唯一重建。下面同時印出時長、這條界線，以及一個440 Hz週期中有多少樣本。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
samples = 1600
for rate in [16000, 8000]:
    print("取樣率", rate, "時長", samples / rate, "奈奎斯特界線", rate / 2, "440Hz每週期樣本", round(rate / 440, 2))

程式並列兩種取樣率。時長是 `樣本數／取樣率`；440 Hz 每週期的樣本數是 `取樣率／440`，約 36.36 與 18.18。後者是假設兩份錄音各自正確記錄同一個 440 Hz 聲音，並不是把原陣列改個標籤。

若原本按 16000 Hz 記錄的數字，直接以 8000 Hz 播放，同樣週期會花兩倍時間，440 Hz 聽起來變 220 Hz。正確改取樣率需要重取樣。降低取樣率時，先把新奈奎斯特界線以上的成分減弱或濾除，這種保留低頻、壓低高頻的動作叫低通處理；否則它們可能在新樣本中混成假的低頻。接著在新的時間點計算數值，維持時長與保留下來的成分的頻率。

提高取樣率時，新界線較高，不必因此刪除原本可表示的頻率；新增樣本只是改用較密的時間點，不會找回錄音時已失去的高頻資訊。

增加模型大小不能把已混在一起的資訊可靠分開；先按上述界線處理原聲音，才談改取樣率。

真人錄音資料因此要先核對取樣率與振幅尺度，不能只看副檔名。例如將 4591 點的 8000 Hz 聲音重取樣成 9182 點的 16000 Hz 聲音，兩者時長仍為 0.573875 秒。完整讀檔格式和既有數字錄音紀錄放在補充，這裡先掌握時間換算。

練習：8000 點、8000 Hz 是 1 秒；若改為 16000 Hz 並維持同一秒，應有多少點？答案 16000，不能只修改檔案中的 rate。

<details>
<summary>回顧與查證</summary>

可回顧：[12.1的波形樣本](https://birdhackor.github.io/tiny-perceptron-vlm/12.1.html)。

原實驗的資料切分、更新設定與逐題結果見[既有實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/real_modal.json)；重做入口見[實驗說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/README.md)。這是上述有限任務的歷史紀錄。

</details>
